In [1]:
import torch_geometric
import pandas as pd
from datetime import datetime


In [2]:
print(torch_geometric.__version__)

2.7.0


In [3]:
file_size = "Small"

In [4]:
ACC_FILES = f"archive/HI-{file_size}_accounts.csv"
TXN_FILES = f"archive/HI-{file_size}_Trans.csv"


In [5]:
DF_TXN = pd.read_csv(TXN_FILES)

In [6]:
DF_TXN

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,US Dollar,3697.340000,US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,US Dollar,0.010000,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,US Dollar,14675.570000,US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,US Dollar,2806.970000,US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,US Dollar,36682.970000,US Dollar,Reinvestment,0
...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,Bitcoin,0.154978,Bitcoin,Bitcoin,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,Bitcoin,0.108128,Bitcoin,Bitcoin,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,Bitcoin,0.004988,Bitcoin,Bitcoin,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,Bitcoin,0.038417,Bitcoin,Bitcoin,0


In [7]:
DF_TXN.dtypes

Timestamp              object
From Bank               int64
Account                object
To Bank                 int64
Account.1              object
Amount Received       float64
Receiving Currency     object
Amount Paid           float64
Payment Currency       object
Payment Format         object
Is Laundering           int64
dtype: object

In [8]:
from typing import Tuple


def string_to_timestamp(date_str: str, date_format: str = "%Y/%m/%d %H:%M") -> Tuple[int,int,int,int,int,int,int]:
    try:
        # Parse the date string into a datetime object
        dt = datetime.strptime(date_str, date_format)
        return dt.year, dt.month, dt.day, dt.weekday(), dt.hour, dt.minute, int(dt.timestamp())
    except Exception as e:
        print(e,date_str)
        raise e

In [9]:
(
    DF_TXN["TS_Year"],
    DF_TXN["TS_Month"],
    DF_TXN["TS_Day"],
    DF_TXN["TS_WeekDay"],
    DF_TXN["TS_Hour"],
    DF_TXN["TS_Minute"],
    DF_TXN["TS_Timestamp"],
) = zip(*DF_TXN["Timestamp"].apply(lambda x: string_to_timestamp(x)))
DF_TXN.drop(columns=['Timestamp'], inplace=True)

In [10]:
DF_TXN.dtypes

From Bank               int64
Account                object
To Bank                 int64
Account.1              object
Amount Received       float64
Receiving Currency     object
Amount Paid           float64
Payment Currency       object
Payment Format         object
Is Laundering           int64
TS_Year                 int64
TS_Month                int64
TS_Day                  int64
TS_WeekDay              int64
TS_Hour                 int64
TS_Minute               int64
TS_Timestamp            int64
dtype: object

In [11]:
if DF_TXN.dtypes['Receiving Currency'].name != 'int64':
    i = 0
    curr_mapper = {}
    for curr_type in set(DF_TXN['Receiving Currency'].unique()).union(set(DF_TXN['Payment Currency'].unique())):
        curr_mapper[curr_type] = i
        i+=1
    DF_TXN['Receiving Currency'] = DF_TXN['Receiving Currency'].apply(lambda x: curr_mapper[x])
    DF_TXN['Payment Currency'] = DF_TXN['Payment Currency'].apply(lambda x: curr_mapper[x])
    DF_TXN['Same_Currency'] = DF_TXN['Receiving Currency'] == DF_TXN['Payment Currency']
    DF_TXN['Same_Currency'] = DF_TXN['Same_Currency'].astype(int)

In [12]:
if DF_TXN.dtypes['Payment Format'].name != 'int64':
    i = 0
    pyment_mapper = {}
    for p_type in set(DF_TXN['Payment Format'].unique()):
        pyment_mapper[p_type] = i
        i+=1
    DF_TXN['Payment Format'] = DF_TXN['Payment Format'].apply(lambda x: pyment_mapper[x])

In [13]:
if 'FROM' not in DF_TXN.columns:
    DF_TXN['FROM'] = DF_TXN[['From Bank','Account']].apply(lambda x: f"{x[0]}>>{x[1]}", axis=1)
if 'TO' not in DF_TXN.columns:
    DF_TXN['TO'] = DF_TXN[['To Bank','Account.1']].apply(lambda x: f"{x[0]}>>{x[1]}", axis=1)
DF_TXN['Same_Bank'] = DF_TXN['FROM'] == DF_TXN['TO']
DF_TXN['Same_Bank'] = DF_TXN['Same_Bank'].astype(int)
for col_to_drop in ['From Bank','Account','To Bank','Account.1']:
    DF_TXN.drop(columns=[col_to_drop], inplace=True)   

C:\Users\Nikhi\AppData\Local\Temp\ipykernel_9956\3873828415.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DF_TXN['FROM'] = DF_TXN[['From Bank','Account']].apply(lambda x: f"{x[0]}>>{x[1]}", axis=1)
C:\Users\Nikhi\AppData\Local\Temp\ipykernel_9956\3873828415.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DF_TXN['TO'] = DF_TXN[['To Bank','Account.1']].apply(lambda x: f"{x[0]}>>{x[1]}", axis=1)


In [14]:
TRAIN_TEST_SPLIT = 0.8
DF_TXN.sort_values(by='TS_Timestamp',inplace=True,ignore_index=True)
N = len(DF_TXN.index)
index_split = int(N*0.8)
DF_TXN_TRAIN = DF_TXN.iloc[0:index_split,:].reset_index(drop=True)
DF_TXN_TEST = DF_TXN.iloc[index_split:,:].reset_index(drop=True)
del DF_TXN

In [15]:
DF_TXN_TRAIN.to_parquet(f"processed_data/HI-{file_size}_Trans_TRAIN_processed.parquet", index=None)

In [16]:
DF_TXN_TEST.to_parquet(f"processed_data/HI-{file_size}_Trans_TEST_processed.parquet",index=None)

In [17]:
import json

In [18]:
with open("processed_data/currency_map.json", 'w+') as f:
    json.dump(curr_mapper, f, indent=4)
with open("processed_data/payment_map.json", 'w+') as f:
    json.dump(pyment_mapper, f, indent=4)